# AmberToolsのtleapを用いたペプチド(Deca-alanine)のモデリング

[AmberTools](https://ambermd.org/AmberTools.php)のモジュールである `tleap` を使用して、10残基のアラニン（デカアラニン）がつながったαヘリックス構造を構築します。


このNotebookで実施する主な行程は以下のとおりです:
1. **環境設定:** AmberToolsのインストール、入出力用のディレクトリを作成。
2. **インプット作成:** tleapのインプットファイル (`.in`) を作成。
3. **ツールの実行:** tleapを実行してパラメータファイル (`.prmtop`, `.inpcrd`) を作り、最後にPDB形式に変換。


## Step1: 環境設定

### Step1-1: AmberToolsの準備

このNotebookでは**AmberTools**に含まれる`tleap`というモジュールを用いてペプチドのモデリングを行います。
AmberToolsがお手元にインストールされれていない場合、以下の手順を参考にインストールしてください。
```bash
conda create --name AmberTools        # 仮想環境の作成
conda activate AmberTools             # 環境のアクティベート
conda install conda-forge::ambertools # AmberToolsのインストール
conda deactivate
```

インストール後、AmberToolsをご使用の際には`amber.sh`を実行して、環境変数を設定します。
```bash
conda activate AmberTools
source $CONDA_PREFIX/amber.sh
    cd to some working folder, and run AmberTools programs
conda deactivate
```

詳細については[Amber公式サイト](https://ambermd.org/GetAmber.php)を参照してください。

### Step1-2: 作業ディレクトリの準備

入力ファイルを入れる `input` と、結果を入れる `output` ディレクトリを作成します

In [ ]:
import os

# ディレクトリのパスを変数で定義
out_dir = "./output/01_modeling"
inp_dir = "./input/01_modeling"

# ディレクトリが存在しない場合のみ作成
os.makedirs(out_dir, exist_ok=True)
os.makedirs(inp_dir, exist_ok=True)

## Step2: tleapのインプットファイル作成

TLeapを動かすためのインプットスクリプトを作成します。

**主な設定内容:**
* **source:** タンパク質用の力場パラメータ（ff14SB）を読み込みます。
* **sequence:** アミノ酸配列を定義します。両端には化学的なキャップ（ACE: アセチル基, NME: メチル基）を付加します。
* **impose:** 二面角（$\phi, \psi$）を特定の値に指定することで、αヘリックス構造を作ります。
    * $\phi$ (Phi): -57.0度 (定義原子: C-N-CA-C)
    * $\psi$ (Psi): -47.0度 (定義原子: N-CA-C-N)

In [2]:
tleap_commands = f"""
# 1. 力場のロード
source leaprc.protein.ff14SB

# 2. 配列の作成
#    ACE: N末端キャップ (Acetyl)
#    NME: C末端キャップ (N-methyl)
mol = sequence {{ ACE ALA ALA ALA ALA ALA ALA ALA ALA ALA ALA NME }}

# 3. αヘリックス構造の適用 (Impose)
#    対象: 残基2-11 (両端のキャップを除くアラニン部分)
#    Phi(φ): C-N-CA-C = -57.0度
#    Psi(ψ): N-CA-C-N = -47.0度
impose mol {{ 2 3 4 5 6 7 8 9 10 11 }} {{ {{ "C" "N" "CA" "C" -57.0 }} {{ "N" "CA" "C" "N" -47.0 }} }}

# 4. パラメータファイル (prmtop と inpcrd) を保存
saveAmberParm mol {out_dir}/deca_alanine.prmtop {out_dir}/deca_alanine.inpcrd

# 終了
quit
"""

# ファイルへの書き出し
filename = f"{inp_dir}/tleap_deca_alanine_helix.in"
with open(filename, "w") as f:
    f.write(tleap_commands)

## Step3: tleapの実行とPDB形式への変換

作成した入力ファイルを使って `tleap` を実行します。
その後、生成されたAMBER形式のファイルを、 PDB形式に変換します。

* `tleap`: パラメータファイルを作成するツール。
* `ambpdb`: AMBER形式のトポロジーと座標からPDBファイルを作成するツール。

In [3]:
# tleapの実行
!tleap -f {inp_dir}/tleap_deca_alanine_helix.in > {out_dir}/tleap.log && mv leap.log {out_dir}

# ambpdbを使ってPDBファイルを生成
!ambpdb -p {out_dir}/deca_alanine.prmtop < {out_dir}/deca_alanine.inpcrd > {out_dir}/deca_alanine_helix.pdb

## Step 4: ファイルの確認

正しくファイルが生成されたか確認しましょう。以下のファイルが `output` ディレクトリにできていれば成功です。

* `deca_alanine.prmtop`: 力場パラメーターファイル
* `deca_alanine.inpcrd`: 座標ファイル
* `deca_alanine_helix.pdb`: 構造ファイル

## Next Step
これで、αヘリックス構造を持つDeca-Alanineのモデリングが完了しました。
次の、[02_equilibrium_nvt_md_ja.ipynb](./02_equilibrium_nvt_md_ja.ipynb)では、分子動力学シミュレーションを使って、モデリングで作成した構造を自然な状態へ緩和させていきます。